In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Rescaling
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import random

np.random.seed(42)
random.seed(42)

In [33]:
def create_data_generators(X_train, X_val, X_test, y_train, y_val, y_test, batch_size=32):
    """
    Crea generadores de datos con augmentation dinámica solo para el conjunto de entrenamiento
    """
    print(f"Creando generadores de datos con batch_size={batch_size}")
    
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=30,
        width_shift_range=0.2,
        height_shift_range=0.2,
        zoom_range=0.2,
        shear_range=0.2,
        horizontal_flip=True,
        brightness_range=[0.8,1.2],
        fill_mode='nearest'
    )
    
    val_test_datagen = ImageDataGenerator(rescale=1./255)
    
    train_generator = train_datagen.flow(
        X_train, y_train,
        batch_size=batch_size,
        shuffle=True
    )
    
    val_generator = val_test_datagen.flow(
        X_val, y_val,
        batch_size=batch_size,
        shuffle=False
    )
    
    test_generator = val_test_datagen.flow(
        X_test, y_test,
        batch_size=batch_size,
        shuffle=False
    )
    
    return train_generator, val_generator, test_generator


In [34]:
def preprocess_pipeline(data_dir, img_size=(224, 224), batch_size=32):
    """Pipeline completo de preprocesamiento de datos"""
        
    print(f"\nIniciando pipeline de preprocesamiento...")
    print(f"Directorio de datos: {data_dir}")
    print(f"Tamaño de imagen: {img_size}")
    print(f"Batch size: {batch_size}\n")

    X, y, classes = load_and_preprocess_data(data_dir, img_size, normalize=False)
    

    y = to_categorical(y, num_classes=len(classes))
    
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=0.3,  # 30% para val+test
        random_state=42,
        stratify=y
    )
    
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.33,  # 10% del total para test
        random_state=42,
        stratify=y_temp
    )
    
    print(f"División de datos:")
    print(f"- Train: {X_train.shape[0]} muestras")
    print(f"- Validación: {X_val.shape[0]} muestras")
    print(f"- Test: {X_test.shape[0]} muestras")
    
    train_generator, val_generator, test_generator = create_data_generators(
        X_train, X_val, X_test,
        y_train, y_val, y_test,
        batch_size
    )
    
    print("\nPipeline de preprocesamiento completado con éxito!")

    return {
        'train_generator': train_generator,
        'val_generator': val_generator,
        'test_generator': test_generator,
        'classes': classes
    }

In [35]:
def load_and_preprocess_data(data_dir, img_size=(224, 224), normalize=False):
    """Carga y preprocesa imágenes desde el directorio de datos"""
    if not os.path.exists(data_dir):
        raise ValueError(f"El directorio {data_dir} no existe")
        
    X = []
    y = []
    
    classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    if not classes:
        raise ValueError(f"No se encontraron subdirectorios (clases) en {data_dir}")
    
    class_indices = {cls: i for i, cls in enumerate(classes)}
    
    print(f"Encontradas {len(classes)} clases: {classes}")
    
    total_images = 0
    for class_name in classes:
        class_dir = os.path.join(data_dir, class_name)
        class_idx = class_indices[class_name]
        
        image_files = [f for f in os.listdir(class_dir) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        print(f"Procesando clase '{class_name}': {len(image_files)} imágenes encontradas")
        
        for img_file in image_files:
            img_path = os.path.join(class_dir, img_file)
            try:
                img = cv2.imread(img_path)
                if img is None:
                    print(f"Error: No se pudo cargar la imagen {img_path}")
                    continue
                
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, img_size)
                
                X.append(img)
                y.append(class_idx)
                total_images += 1

            except Exception as e:
                print(f"Error procesando {img_path}: {str(e)}")
    
    if total_images == 0:
        raise ValueError("No se pudieron cargar imágenes del directorio")
        
    print(f"Total de imágenes cargadas: {total_images}")
    
    X = np.array(X)
    y = np.array(y)

    print(f"Forma del array X: {X.shape}")
    print(f"Forma del array y: {y.shape}")
    
    if normalize:
        X = X.astype('float32') / 255.0
    
    return X, y, classes


In [ ]:
data_dir = "/Users/andres/proyecto_final/Real-Time-Spanish-Sign-Language-Recognition/datasets/SSLdictionary"
generators = preprocess_pipeline(data_dir, img_size=(224, 224), batch_size=32)

## Para usar el código en el entrenamiento

In [39]:
def create_model(input_shape, num_classes):
    """
    Crea un modelo CNN para clasificación de señas
    """
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D(2, 2),
        
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        
        Flatten(),
        
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [38]:
img_size = (224, 224)
batch_size = 32
epochs = 50

In [ ]:
generators = preprocess_pipeline(data_dir, img_size=img_size, batch_size=batch_size)
input_shape = (img_size[0], img_size[1], 3)  # (224, 224, 3)
num_classes = len(generators['classes'])
model = create_model(input_shape, num_classes)
    

In [ ]:
model.summary()

In [ ]:
history = model.fit(
    generators['train_generator'],
    validation_data=generators['val_generator'],
    epochs=epochs,
    steps_per_epoch=len(generators['train_generator']),
    validation_steps=len(generators['val_generator']))
    
test_loss, test_accuracy = model.evaluate(generators['test_generator'])
print(f"\nPrecisión en el conjunto de prueba: {test_accuracy*100:.2f}%")
    

In [ ]:
plt.figure(figsize=(12, 4))
    
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Precisión del modelo')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida del modelo')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
model.save('modelo_lenguaje_señas_CNN.h5')

# MobileNetV2

In [58]:
def load_and_preprocess_data(data_dir, img_size=(224, 224)):
    """Carga y preprocesa imágenes desde el directorio de datos"""
    if not os.path.exists(data_dir):
        raise ValueError(f"El directorio {data_dir} no existe")
        
    X = []
    y = []
    
    classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    if not classes:
        raise ValueError(f"No se encontraron subdirectorios (clases) en {data_dir}")
    
    class_indices = {cls: i for i, cls in enumerate(classes)}
    
    print(f"Encontradas {len(classes)} clases: {classes}")
    
    total_images = 0
    for class_name in classes:
        class_dir = os.path.join(data_dir, class_name)
        class_idx = class_indices[class_name]
        
        image_files = [f for f in os.listdir(class_dir) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        print(f"Procesando clase '{class_name}': {len(image_files)} imágenes encontradas")
        
        for img_file in image_files:
            img_path = os.path.join(class_dir, img_file)
            try:
                img = cv2.imread(img_path)
                if img is None:
                    print(f"Error: No se pudo cargar la imagen {img_path}")
                    continue
                
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, img_size)
                
                X.append(img)
                y.append(class_idx)
                total_images += 1

            except Exception as e:
                print(f"Error procesando {img_path}: {str(e)}")
    
    if total_images == 0:
        raise ValueError("No se pudieron cargar imágenes del directorio")
        
    print(f"Total de imágenes cargadas: {total_images}")
    
    X = np.array(X)
    y = np.array(y)

    print(f"Forma del array X: {X.shape}")
    print(f"Forma del array y: {y.shape}")
    
    return X, y, classes


In [71]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomContrast(0.2),
])

In [60]:
def create_datasets(X_train, X_val, X_test, y_train, y_val, y_test, batch_size=32):
    """Crea tf.data.Dataset con augmentation y optimización"""
    
    def process_train_data(image, label):
        image = tf.cast(image, tf.float32)
        image = data_augmentation(image, training=True)
        image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
        return image, label
    
    def process_val_test_data(image, label):
        image = tf.cast(image, tf.float32)
        image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
        return image, label
        
    train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
    
    train_ds = train_ds.map(process_train_data, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    val_ds = val_ds.map(process_val_test_data, num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    test_ds = test_ds.map(process_val_test_data, num_parallel_calls=tf.data.AUTOTUNE)
    test_ds = test_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    return train_ds, val_ds, test_ds

In [61]:
def create_model(input_shape, num_classes):
    """Crea un modelo basado en MobileNetV2 con transfer learning"""
    base_model = MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    
    base_model.trainable = False
    
    inputs = tf.keras.Input(shape=input_shape)
    
    x = base_model(inputs, training=False)
    
    x = GlobalAveragePooling2D()(x)
    
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = tf.keras.Model(inputs, outputs)
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model, base_model


In [62]:
def preprocess_pipeline(data_dir, img_size=(224, 224), batch_size=32):
    """Pipeline completo de preprocesamiento de datos"""
    print(f"\nIniciando pipeline de preprocesamiento...")
    print(f"Directorio de datos: {data_dir}")
    print(f"Tamaño de imagen: {img_size}")
    print(f"Batch size: {batch_size}\n")

    X, y, classes = load_and_preprocess_data(data_dir, img_size)
    
    y_one_hot = tf.keras.utils.to_categorical(y, num_classes=len(classes))
    
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y_one_hot,
        test_size=0.3,  # 30% para val+test
        random_state=42,
        stratify=y
    )
    
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.33,  # 10% del total para test
        random_state=42,
        stratify=y_temp
    )
    
    print(f"División de datos:")
    print(f"- Train: {X_train.shape[0]} muestras")
    print(f"- Validación: {X_val.shape[0]} muestras")
    print(f"- Test: {X_test.shape[0]} muestras")
    
    train_ds, val_ds, test_ds = create_datasets(
        X_train, X_val, X_test,
        y_train, y_val, y_test,
        batch_size
    )
    
    print("\nPipeline de preprocesamiento completado con éxito!")

    return {
        'train_ds': train_ds,
        'val_ds': val_ds,
        'test_ds': test_ds,
        'classes': classes,
        'steps_per_epoch': len(X_train) // batch_size,
        'validation_steps': len(X_val) // batch_size
    }

In [ ]:
data_dir = "/Users/andres/proyecto_final/Real-Time-Spanish-Sign-Language-Recognition/datasets/SSLdictionary"
img_size = (224, 224)
batch_size = 32
initial_epochs = 10
fine_tuning_epochs = 10

data = preprocess_pipeline(data_dir, img_size=img_size, batch_size=batch_size)

input_shape = (img_size[0], img_size[1], 3)
num_classes = len(data['classes'])
model, base_model = create_model(input_shape, num_classes)

model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-6
    )
]

print("Entrenamiento inicial con base model congelado...")
history = model.fit(
    data['train_ds'],
    validation_data=data['val_ds'],
    epochs=initial_epochs,
    callbacks=callbacks
)

In [ ]:
print("Fine-tuning del modelo...")
base_model.trainable = True
    
for layer in base_model.layers[:100]:
    layer.trainable = False
    
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
    
fine_tuning_history = model.fit(
    data['train_ds'],
    validation_data=data['val_ds'],
    epochs=initial_epochs + fine_tuning_epochs,
    initial_epoch=initial_epochs,
    steps_per_epoch=data['steps_per_epoch'],
    validation_steps=data['validation_steps']
)
    

In [ ]:
test_results = model.evaluate(data['test_ds'])
test_loss, test_accuracy = test_results
print(f"\nPrecisión en el conjunto de prueba: {test_accuracy*100:.2f}%")
    
acc = history.history['accuracy'] + fine_tuning_history.history['accuracy']
val_acc = history.history['val_accuracy'] + fine_tuning_history.history['val_accuracy']
loss = history.history['loss'] + fine_tuning_history.history['loss']
val_loss = history.history['val_loss'] + fine_tuning_history.history['val_loss']    

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(acc, label='Entrenamiento')
plt.plot(val_acc, label='Validación')
plt.plot([initial_epochs-1, initial_epochs-1],
         plt.ylim(), label='Inicio Fine Tuning')
plt.title('Precisión del modelo')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label='Entrenamiento')
plt.plot(val_loss, label='Validación')
plt.plot([initial_epochs-1, initial_epochs-1],
         plt.ylim(), label='Inicio Fine Tuning')
plt.title('Pérdida del modelo')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
model.save('modelo_lenguaje_señas_mobilenetv2.h5')

Vamos a intentar aplicar estas mejoras para mejorar la precisión del 88.89% pasada

* Mejorar el data augmentation, añadiendo más transformaciones, variedad y normalización.

* Fine-tuning más profundio, descongelando las últimas 20 capas del modelo base y utilizar una tasa de aprendizaje menor para evitar sobre ajustar las capas pre-entrenadas

* Mejorar la arquitectura, añadiendo capas denas más complejas, con mayor dropout, regularización L2 y ReduceLROnPlateau para reducir la tasa de aprendizaje cuando la mejora se estanque.


In [4]:
import tensorflow.keras.layers as layers

augmentation_layers = tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.2),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomZoom(0.2),
        layers.RandomContrast(0.2),
        layers.RandomBrightness(0.2),
        layers.GaussianNoise(0.1),
        layers.Resizing(224, 224),
        layers.Rescaling(1./127.5, offset=-1)
    ])

def process_train_data(image, label):
    """Procesa datos de entrenamiento con augmentación"""
    image = tf.cast(image, tf.float32)
    image = augmentation_layers(image)
    return image, label

def process_val_test_data(image, label):
    """Procesa datos de validación/test"""
    image = tf.cast(image, tf.float32)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label


def create_datasets(X_train, X_val, X_test, y_train, y_val, y_test, batch_size):
    """Crea los datasets de entrenamiento, validación y prueba"""
    train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
    
    train_ds = train_ds.map(process_train_data, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    val_ds = val_ds.map(process_val_test_data, num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    test_ds = test_ds.map(process_val_test_data, num_parallel_calls=tf.data.AUTOTUNE)
    test_ds = test_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    return train_ds, val_ds, test_ds

def create_model(input_shape, num_classes):
    """Crea un modelo MobileNetV2 mejorado"""
    base_model = MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    
    for layer in base_model.layers[:-15]:
        layer.trainable = False
    
    inputs = tf.keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    
    outputs = layers.Dense(num_classes, activation='softmax', 
                         kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    
    model = tf.keras.Model(inputs, outputs)
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model, base_model


def train_model(model, data, initial_epochs=15):
    """Entrena el modelo con callbacks optimizados"""
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', 
        factor=0.2,
        patience=3,
        min_lr=0.00001
    )
    
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True
    )
    
    print("\nEntrenando el modelo...")
    history = model.fit(
        data['train_ds'],
        epochs=initial_epochs,
        validation_data=data['val_ds'],
        callbacks=[reduce_lr, early_stopping]
    )
    
    # Evaluar
    test_loss, test_acc = model.evaluate(data['test_ds'])
    print(f"\nPrecisión en el conjunto de prueba: {test_acc:.2%}")
    
    return history

def preprocess_pipeline(data_dir, img_size=(224, 224), batch_size=32):
    """
    Pipeline completo de preprocesamiento de datos
    """
    if not os.path.exists(data_dir):
        raise ValueError(f"El directorio {data_dir} no existe")

    images = []
    labels = []
    
    classes = [d for d in sorted(os.listdir(data_dir)) 
              if os.path.isdir(os.path.join(data_dir, d))]
    
    if not classes:
        raise ValueError(f"No se encontraron subdirectorios (clases) en {data_dir}")
    
    print(f"Encontradas {len(classes)} clases: {classes}")
    
    total_images = 0
    for idx, class_name in enumerate(classes):
        class_dir = os.path.join(data_dir, class_name)
        
        image_files = [f for f in os.listdir(class_dir) 
                      if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        
        print(f"Procesando clase '{class_name}': {len(image_files)} imágenes encontradas")
        
        for img_name in image_files:
            img_path = os.path.join(class_dir, img_name)
            try:
                img = tf.keras.preprocessing.image.load_img(
                    img_path,
                    target_size=img_size
                )
                img_array = tf.keras.preprocessing.image.img_to_array(img)
                
                images.append(img_array)
                labels.append(idx)
                total_images += 1

            except Exception as e:
                print(f"Error procesando {img_path}: {str(e)}")
    
    if total_images == 0:
        raise ValueError("No se pudieron cargar imágenes del directorio")
        
    print(f"Total de imágenes cargadas: {total_images}")
    
    X = np.array(images)
    y = np.array(labels)

    print(f"Forma del array X: {X.shape}")
    print(f"Forma del array y: {y.shape}")
    
    y = tf.keras.utils.to_categorical(y, num_classes=len(classes))
    
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)
    
    print(f"\nDivisión del dataset completada:")
    print(f"- Train: {X_train.shape[0]} muestras")
    print(f"- Validación: {X_val.shape[0]} muestras")
    print(f"- Test: {X_test.shape[0]} muestras")
    
    train_ds, val_ds, test_ds = create_datasets(
        X_train, X_val, X_test,
        y_train, y_val, y_test,
        batch_size
    )
    
    print("\nPipeline de preprocesamiento completado con éxito!")
    
    return {
        'train_ds': train_ds,
        'val_ds': val_ds,
        'test_ds': test_ds,
        'classes': classes,
        'validation_steps': len(X_val) // batch_size
    }

In [5]:
img_size = (224, 224)
batch_size = 32
data_dir = "/Users/andres/proyecto_final/Real-Time-Spanish-Sign-Language-Recognition/datasets/SSLdictionary"
    
data = preprocess_pipeline(data_dir, img_size=img_size, batch_size=batch_size)
input_shape = (img_size[0], img_size[1], 3)
num_classes = len(data['classes'])
model, base_model = create_model(input_shape, num_classes)

    
model.summary()
    
history = train_model(model, data)
    

Encontradas 19 clases: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U']
Procesando clase 'A': 102 imágenes encontradas
Procesando clase 'B': 95 imágenes encontradas
Procesando clase 'C': 98 imágenes encontradas
Procesando clase 'D': 102 imágenes encontradas
Procesando clase 'E': 103 imágenes encontradas
Procesando clase 'F': 105 imágenes encontradas
Procesando clase 'G': 108 imágenes encontradas
Procesando clase 'I': 113 imágenes encontradas
Procesando clase 'K': 108 imágenes encontradas
Procesando clase 'L': 112 imágenes encontradas
Procesando clase 'M': 114 imágenes encontradas
Procesando clase 'N': 111 imágenes encontradas
Procesando clase 'O': 97 imágenes encontradas
Procesando clase 'P': 110 imágenes encontradas
Procesando clase 'Q': 120 imágenes encontradas
Procesando clase 'R': 90 imágenes encontradas
Procesando clase 'S': 100 imágenes encontradas
Procesando clase 'T': 102 imágenes encontradas
Procesando clase 'U': 108 imágenes enco

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 19)             │         4,883 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,050,067 (11.64 MB)

 Trainable params: 1,832,083 (6.99 MB)

 Non-trainable params: 1,217,984 (4.65 MB)


Entrenando el modelo...
Epoch 1/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 18s 389ms/step - accuracy: 0.0832 - loss: 3.0302 - val_accuracy: 0.3406 - val_loss: 2.5742 - learning_rate: 1.0000e-04
Epoch 2/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 15s 369ms/step - accuracy: 0.3567 - loss: 2.2219 - val_accuracy: 0.4812 - val_loss: 1.9965 - learning_rate: 1.0000e-04
Epoch 3/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 15s 375ms/step - accuracy: 0.5390 - loss: 1.5660 - val_accuracy: 0.6594 - val_loss: 1.3511 - learning_rate: 1.0000e-04
Epoch 4/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 16s 383ms/step - accuracy: 0.6706 - loss: 1.0560 - val_accuracy: 0.7500 - val_loss: 0.9391 - learning_rate: 1.0000e-04
Epoch 5/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 15s 373ms/step - accuracy: 0.7581 - loss: 0.7938 - val_accuracy: 0.7594 - val_loss: 0.8102 - learning_rate: 1.0000e-04
Epoch 6/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 15s 372ms/step - accuracy: 0.8283 - loss: 0.5979 - val_accuracy: 0.8625 - val_loss: 0.5415 - learning_rate: 1.0000e-04
Epoch 7/15
40/40 ━━━━━━━━━━━━━━━━

In [6]:
model.save("modelo_mejorado_mobilenetv2.h5")